In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, fbeta_score

In [2]:
# ==========================================
# 1. Load Training Data ONLY
# ==========================================
# We only load train.csv. The test.csv remains untouched.
train_df = pd.read_csv('../data/processed/train.csv')

In [3]:
X_train = train_df.drop(columns=['Diabetes_01'])
y_train = train_df['Diabetes_01']

In [4]:
# ==========================================
# 2. Setup Stratified Cross-Validation
# ==========================================
# Define the 5-fold cross-validation blueprint
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
# ==========================================
# 3. Initialize Dummy Classifier (Baseline)
# ==========================================
# 'most_frequent' means the model simply guesses the majority class every time
dummy_clf = DummyClassifier(strategy='most_frequent', random_state=42)

In [6]:
# ==========================================
# 4. Evaluate using manual cross-validation
# ==========================================
print("--- Evaluating Dummy Classifier Baseline ---\n")
fold_scores = []

for train_idx, valid_idx in cv_strategy.split(X_train, y_train):
    estimator = clone(dummy_clf)

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_valid = y_train.iloc[valid_idx]

    estimator.fit(X_fold_train, y_fold_train)
    y_valid_pred = estimator.predict(X_valid)

    fold_scores.append({
        "accuracy": accuracy_score(y_valid, y_valid_pred),
        "recall": recall_score(y_valid, y_valid_pred, average="macro", zero_division=0),
        "f1": f1_score(y_valid, y_valid_pred, average="macro", zero_division=0),
        "f2": fbeta_score(y_valid, y_valid_pred, beta=2, average="macro", zero_division=0),
    })

fold_scores_df = pd.DataFrame(fold_scores)

print(f"Mean Accuracy: {fold_scores_df['accuracy'].mean():.4f}")
print(f"Mean Recall:   {fold_scores_df['recall'].mean():.4f}")
print(f"Mean F1-Score: {fold_scores_df['f1'].mean():.4f}")
print(f"Mean F2-Score: {fold_scores_df['f2'].mean():.4f}")

--- Evaluating Dummy Classifier Baseline ---

Mean Accuracy: 0.8473
Mean Recall:   0.5000
Mean F1-Score: 0.4587
Mean F2-Score: 0.4826
